# Predicting Medication Non-Adherence Using Medicare Part D Data

## Project Overview

Medication non-adherence is a challenge in healthcare, potentially contributing to poor patient outcomes, avoidable hospitalizations, and increased healthcare costs.

This project develops an end-to-end machine learning pipeline to identify beneficiaries at risk of medication non-adherence using prescription drug event data.

### Project Objectives

1. Analyze historical prescription refill patterns.
2. Engineer features related to refill gaps, medication burden, and prescription costs.
3. Calculate medication adherence using Proportion of Days Covered (PDC).
4. Develop predictive models using Logistic Regression and Gradient Boosting.
5. Evaluate model performance using precision, recall, F1-score, and ROC-AUC.
6. Explore potential behavioral factors associated with medication non-adherence.

### Data Source

CMS Medicare Part D Prescription Drug Events (PDE)

Development dataset: CMS 2008–2010 Data Entrepreneurs' Synthetic Public Use File (DE-SynPUF).

The synthetic data is intended for pipeline development and education, not validated clinical inference.

### Technologies

- Python
- Pandas and NumPy
- SQL (DuckDB)
- Scikit-Learn
- Matplotlib and Seaborn

### Prediction Target

Use 180 days of historical prescription data to predict whether a beneficiary will have a PDC below 80% during the following 90 days.

This is a custom predictive target informed by CMS adherence methodology, not an exact CMS Star Ratings measure.


In [2]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine, text
import psycopg

print("Python environment configured successfully!")
print("Pandas:", pd.__version__)
print("Scikit-Learn:", sklearn.__version__)
print("Psycopg:", psycopg.__version__)


Python environment configured successfully!
Pandas: 3.1.1
Scikit-Learn: 1.9.1
Psycopg: 3.3.9


## 2. Database Connectivity

### Objective
Establish a connection between the Python environment and the PostgreSQL database hosted in Docker.

**Database Configuration**
- Database: medicare_project
- Database Engine: PostgreSQL 18.3
- Host: localhost
- Port: 5432
- Connection Library: SQLAlchemy with Psycopg 3

This connection will be used to extract, transform, and analyze Medicare Part D prescription drug event data.


In [4]:
import os
from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Database configuration
DB_USER = "postgres"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "medicare_project"

# Read the password securely
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD") or getpass(
    "Enter PostgreSQL password: "
)

# Construct the connection URL
DATABASE_URL = URL.create(
    drivername="postgresql+psycopg",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

# Create the database engine
engine = create_engine(DATABASE_URL)

# Verify connection
try:
    with engine.connect() as connection:
        result = connection.execute(
            text("""
                SELECT
                    current_database() AS database,
                    current_user AS username,
                    current_setting('server_version') AS version
            """)
        )

        print("PostgreSQL Connection Successful!")
        print(dict(result.mappings().one()))

except Exception as e:
    print("Connection failed:", e)


PostgreSQL Connection Successful!
{'database': 'medicare_project', 'username': 'postgres', 'version': '18.3 (Debian 18.3-1.pgdg13+1)'}
